# TrashFlow — YOLOv8s fine-tune on TACO (5 classes)

Run on Google Colab with T4 GPU (free tier). Expected runtime: ~30 minutes for 50 epochs.

**Outputs**:
- `trash_yolov8s.pt` — PyTorch weights
- `trash_yolov8s.onnx` — CPU-optimized ONNX
- `metrics.json` — per-class precision/recall + mAP@50

**Target**: mAP@50 ≥ 0.55 on TACO val split.

## Prereqs
1. Runtime → Change runtime type → GPU (T4)
2. Roboflow account (free) → https://roboflow.com → create workspace → copy API key from Settings

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime

In [ ]:
# Verify GPU is available
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 1. Download TACO (Trash Annotations in Context) via Roboflow

The TACO dataset has 60 classes in the wild; Roboflow hosts remapped versions. We use the 5-class super-category mapping from Roboflow Universe.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'PASTE_YOUR_KEY_HERE'  # ← from https://app.roboflow.com/settings/api

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# Community project with TACO remapped to 5 classes. Change version if newer exists.
project = rf.workspace('mohamed-traore-2ekkp').project('taco-trash-annotations-in-context')
dataset = project.version(16).download('yolov8')
print(f'Dataset at: {dataset.location}')

In [ ]:
# Inspect dataset structure and class list
import yaml
from pathlib import Path

data_yaml = Path(dataset.location) / 'data.yaml'
with open(data_yaml) as f:
    config = yaml.safe_load(f)

print('Classes in dataset:', config.get('names'))
print('Train images:', len(list(Path(dataset.location, 'train/images').glob('*'))))
print('Val images:  ', len(list(Path(dataset.location, 'valid/images').glob('*'))))

## 2. Remap classes to our 5-class schema

Our canonical order (must match `apps/ml/app/schemas.py`):
0. plastic
1. paper
2. glass
3. metal
4. hazardous

If the Roboflow dataset already uses these exact 5 classes in the same order — skip this cell. Otherwise, update `data.yaml`:

In [ ]:
TARGET_CLASSES = ['plastic', 'paper', 'glass', 'metal', 'hazardous']

with open(data_yaml) as f:
    config = yaml.safe_load(f)

source_names = config.get('names') or []
if source_names == TARGET_CLASSES:
    print('Already in target order — no remap needed.')
else:
    print('Source classes:', source_names)
    print('Target classes:', TARGET_CLASSES)
    print('⚠️  Manual remap required — edit label files to match target indices, or pick a different Roboflow version.')

## 3. Fine-tune from YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # 22MB, good balance speed/accuracy for Railway CPU

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,             # early stop if val mAP doesn't improve for 10 epochs
    project='trashflow',
    name='yolov8s_taco_5class',
    augment=True,
    mixup=0.1,
    copy_paste=0.1,
    device=0,
    workers=2,
    verbose=True,
)

## 4. Validate and record metrics

In [ ]:
import json

weights_path = f'trashflow/yolov8s_taco_5class/weights/best.pt'
model = YOLO(weights_path)

val_results = model.val(data=str(data_yaml), split='val', imgsz=640)

metrics = {
    'mAP50':        float(val_results.box.map50),
    'mAP50-95':     float(val_results.box.map),
    'precision':    float(val_results.box.mp),
    'recall':       float(val_results.box.mr),
    'per_class': {
        name: {
            'precision': float(val_results.box.p[i]) if i < len(val_results.box.p) else None,
            'recall':    float(val_results.box.r[i]) if i < len(val_results.box.r) else None,
            'mAP50':     float(val_results.box.ap50[i]) if i < len(val_results.box.ap50) else None,
        }
        for i, name in enumerate(TARGET_CLASSES)
    },
}

print(json.dumps(metrics, indent=2))

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

assert metrics['mAP50'] >= 0.50, 'mAP@50 below 0.50 — retrain with more epochs or better data'

## 5. Export to ONNX and save artifacts

ONNX gives 1.5–2× speedup on CPU (Railway free tier has no GPU).

In [ ]:
model.export(format='onnx', optimize=True, dynamic=False, simplify=True)

import shutil
shutil.copy(weights_path, 'trash_yolov8s.pt')
shutil.copy(weights_path.replace('.pt', '.onnx'), 'trash_yolov8s.onnx')

# Sanity-check file sizes
import os
for f in ['trash_yolov8s.pt', 'trash_yolov8s.onnx', 'metrics.json']:
    print(f'{f}: {os.path.getsize(f) / 1024 / 1024:.2f} MB')

## 6. Download artifacts

The next step is uploading these three files to Supabase Storage bucket `ml-artifacts` (public), then `wget` from Railway. See `docs/SETUP.md` section 3.4.

In [ ]:
from google.colab import files

files.download('trash_yolov8s.pt')
files.download('trash_yolov8s.onnx')
files.download('metrics.json')

## Smoke test: classify a photo

Upload a trash photo from your phone to verify the model actually works before deploying.

In [ ]:
from google.colab import files as _files
import io
from PIL import Image

uploaded = _files.upload()  # pick any trash photo
for filename, data in uploaded.items():
    img = Image.open(io.BytesIO(data))
    result = model.predict(img, imgsz=640, conf=0.25, verbose=False)[0]

    probs = result.probs
    if probs is not None:
        scores = dict(zip(TARGET_CLASSES, probs.data.tolist()))
        top = max(scores, key=scores.get)
        print(f'{filename} → {top} ({scores[top]:.2%})')
        for cat, score in sorted(scores.items(), key=lambda kv: -kv[1]):
            print(f'  {cat:>10}: {score:.2%}')
    else:
        print(f'{filename}: no probs output (detection model, not classifier)')